# Multi-Agent LLM Research Automation Platform
## Complete AI Agents Documentation

This notebook documents all 35+ AI agents with complete source code, testing, and evaluation.

**Project**: Multi-agent system using LangGraph for automated academic research  
**Agents**: 35+ specialized AI agents across 19 categories  
**Models**: 4 specialized model slots (Reasoning, Writing, Coding, Critical)

---

## Table of Contents
1. Environment Configuration
2. LLM Provider Factory
3. Base Agent Class
4. Agent Registry
5. Agent Implementations by Category
6. Testing Code
7. Evaluation Code
8. System Architecture


## 1. Environment Configuration

In [13]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env')

# Display configuration (masked API keys)
def mask_key(key, show_chars=4):
    value = os.getenv(key, '')
    if value and len(value) > show_chars:
        return value[:show_chars] + '***' + value[-show_chars:]
    return value if value else 'NOT SET'

config_display = {
    'LLM_STATUS': os.getenv('LLM_STATUS', 'OFFLINE'),
    'OLLAMA_BASE_URL': os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434'),
    'MODEL_REASONING': os.getenv('MODEL_REASONING', 'phi3:mini'),
    'MODEL_WRITING': os.getenv('MODEL_WRITING', 'gemma2:2b'),
    'MODEL_CODING': os.getenv('MODEL_CODING', 'qwen2.5-coder:latest'),
    'MODEL_CRITICAL': os.getenv('MODEL_CRITICAL', 'phi3:mini'),
    'MAX_TOKENS': os.getenv('MAX_TOKENS', '4096'),
    'OPENROUTER_API': mask_key('OPENROUTER_API_1'),
    'GROQ_API': mask_key('GROQ_API_1'),
    'GEMINI_API': mask_key('GEMINI_API_1'),
}

import pprint
print("Configuration:")
pprint.pprint(config_display)


Configuration:
{'GEMINI_API': 'AIza***umbU',
 'GROQ_API': 'gsk_***5Flc',
 'LLM_STATUS': 'OFFLINE                            # block 1 had OFFLINE',
 'MAX_TOKENS': '10096                             # block 1 had 4096',
 'MODEL_CODING': 'qwen2.5-coder:latest',
 'MODEL_CRITICAL': 'gemini/gemini-2.0-flash',
 'MODEL_REASONING': 'openrouter/anthropic/claude-3.5-sonnet:beta',
 'MODEL_WRITING': 'groq/gemma2-9b-it',
 'OLLAMA_BASE_URL': 'http://host.docker.internal:11434   # block 1 had '
                    'http://localhost:11434',
 'OPENROUTER_API': 'sk-o***802f'}


### 1.1 LLM Operating Modes

In [14]:
# LLM Operating Modes
llm_modes = {
    'OFFLINE': {'description': 'Use Ollama local models', 'provider': 'OllamaProvider'},
    'ONLINE': {'description': 'Use cloud providers (Groq/OpenRouter/Gemini)'},
    'HYBRID': {'description': 'Prefer cloud, fallback to Ollama'}
}

for mode, details in llm_modes.items():
    print(f"{mode}: {details['description']}")


OFFLINE: Use Ollama local models
ONLINE: Use cloud providers (Groq/OpenRouter/Gemini)
HYBRID: Prefer cloud, fallback to Ollama


---

## 2. Configuration Module (ai_engine/config.py)


In [18]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sys
from pathlib import Path

# Correct path resolution
NOTEBOOK_DIR = Path(__file__).parent if '__file__' in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Only add project root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Safe import
try:
    from ai_engine import config
except Exception as e:
    print("❌ Config import failed:", e)
    raise

# Debug output
print("Configuration Module Loaded:")
print(f"  LLM_STATUS: {config.LLM_STATUS}")
print(f"  MODEL_REASONING: {config.MODEL_REASONING}")
print(f"  MODEL_WRITING: {config.MODEL_WRITING}")
print(f"  MODEL_CODING: {config.MODEL_CODING}")
print(f"  MODEL_CRITICAL: {config.MODEL_CRITICAL}")
print(f"  GROQ_API_KEYS: {len(config.GROQ_API_KEYS)} keys")
print(f"  OPENROUTER_API_KEYS: {len(config.OPENROUTER_API_KEYS)} keys")

# Basic validation
assert config.MAX_TOKENS > 0

[Config] HF_HOME set to: D:\SEM 6\AIML317_PROJECT_III\project_sgp\data\huggingface
[Config] LLM_STATUS = OFFLINE (mode=offline)
[Config] Ollama base URL: http://host.docker.internal:11434   # block 1 had http://localhost:11434
[Config] HF_HOME set to: d:\SEM 6\AIML317_PROJECT_III\project_sgp\data\huggingface
[Config] LLM_STATUS = OFFLINE (mode=offline)
[Config] Ollama base URL: http://host.docker.internal:11434   # block 1 had http://localhost:11434
Configuration Module Loaded:
  LLM_STATUS: OFFLINE
  MODEL_REASONING: openrouter/anthropic/claude-3.5-sonnet:beta
  MODEL_WRITING: groq/gemma2-9b-it
  MODEL_CODING: qwen2.5-coder:latest
  MODEL_CRITICAL: gemini/gemini-2.0-flash
  GROQ_API_KEYS: 3 keys
  OPENROUTER_API_KEYS: 3 keys


---

## 3. LLM Provider Factory


In [19]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.llm.factory import get_llm_provider, get_llm_status, clear_provider_cache

status = get_llm_status()
print("LLM Status:")
pprint.pprint(status)


[Factory] Ollama unreachable. Falling back to Groq.


LLM Status:
{'config': {'groq_default_models': ['llama-3.1-8b-instant',
                                    'llama-3.3-70b-versatile'],
            'max_tokens': 10096,
            'model_coding': 'qwen2.5-coder:latest',
            'model_critical': 'gemini/gemini-2.0-flash',
            'model_reasoning': 'openrouter/anthropic/claude-3.5-sonnet:beta',
            'model_writing': 'groq/gemma2-9b-it'},
 'mode': 'OFFLINE',
 'provider': {'active_key_index': 1,
              'available': True,
              'fallback_models': ['openrouter/anthropic/claude-3.5-sonnet:beta',
                                  'llama-3.1-8b-instant',
                                  'llama-3.3-70b-versatile'],
              'key_rotation': 'round-robin',
              'max_retries': 3,
              'model': 'openrouter/anthropic/claude-3.5-sonnet:beta',
              'model_failures': {'llama-3.1-8b-instant': 0,
                                 'llama-3.3-70b-versatile': 0,
                                

In [20]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Get providers for each model slot
clear_provider_cache()

try:
    from ai_engine import config
    providers = {}
    for slot, model in [('reasoning', config.MODEL_REASONING), 
                         ('writing', config.MODEL_WRITING),
                         ('coding', config.MODEL_CODING),
                         ('critical', config.MODEL_CRITICAL)]:
        try:
            provider = get_llm_provider(model)
            providers[slot] = {'model': model, 'provider': provider.provider_name}
        except Exception as e:
            providers[slot] = {'model': model, 'error': str(e)}
    
    print("Model Slots and Providers:")
    pprint.pprint(providers)
except Exception as e:
    print(f"Provider check error: {e}")


[Factory] Ollama unreachable. Falling back to Groq.
[Factory] Ollama unreachable. Falling back to Groq.
[Factory] Ollama unreachable. Falling back to Groq.
[Factory] Ollama unreachable. Falling back to Groq.


Model Slots and Providers:
{'coding': {'model': 'qwen2.5-coder:latest', 'provider': 'groq'},
 'critical': {'model': 'gemini/gemini-2.0-flash', 'provider': 'groq'},
 'reasoning': {'model': 'openrouter/anthropic/claude-3.5-sonnet:beta',
               'provider': 'groq'},
 'writing': {'model': 'groq/gemma2-9b-it', 'provider': 'groq'}}


---

## 4. Base Agent Class


In [21]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.base import BaseAgent

print("BaseAgent Class:")
print(f"  TOKEN_LIMITS: {BaseAgent.TOKEN_LIMITS}")
print("\nMethods:")
for m in dir(BaseAgent):
    if not m.startswith('_') and callable(getattr(BaseAgent, m)):
        print(f"  - {m}")


BaseAgent Class:
  TOKEN_LIMITS: {'phi3:mini': 4096, 'gemma2:2b': 8192, 'qwen2.5-coder:latest': 4096, 'default': 4096}

Methods:
  - arun
  - run


In [22]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Create test agent
class TestAgent(BaseAgent):
    def __init__(self):
        super().__init__(name="TestAgent", system_prompt="You are a test agent.")

try:
    test_agent = TestAgent()
    print(f"Agent Created: {test_agent.name}")
    print(f"Model: {test_agent.model_name}")
except Exception as e:
    print(f"Note: {e}")
    print("(This is expected if Ollama is not running)")


[Factory] Ollama unreachable. Falling back to Groq.


Agent Created: TestAgent
Model: openrouter/anthropic/claude-3.5-sonnet:beta


In [23]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# JSON Extraction Demo
agent = BaseAgent(name="Demo", system_prompt="Demo")

test_cases = [
    '{"key": "value", "number": 42}',
    '```json\n{"status": "success"}\n```',
    'Answer is {"result": "found"}',
    'Plain text'
]

print("JSON Extraction Tests:")
for i, text in enumerate(test_cases, 1):
    result = agent._extract_json(text)
    print(f"  {i}. {text[:30]}... -> {result}")


[Demo] Could not parse JSON. Returning raw text.


JSON Extraction Tests:
  1. {"key": "value", "number": 42}... -> {'key': 'value', 'number': 42}
  2. ```json
{"status": "success"}
... -> {'status': 'success'}
  3. Answer is {"result": "found"}... -> {'result': 'found'}
  4. Plain text... -> {'raw_text': 'Plain text'}


---

## 5. Agent Registry


In [24]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.registry import AGENTS

print(f"Total Registered Agents: {len(AGENTS)}")
print("\nAgents by Category:")

categories = {
    'Topic Discovery': ['topic_discovery', 'topic_lock'],
    'Orchestration': ['orchestrator'],
    'Discovery': ['domain_intelligence', 'historical_review'],
    'Review': ['slr', 'survey_meta_analysis'],
    'Synthesis': ['gap_synthesis', 'research_question', 'conceptual_framework'],
    'Understanding': ['paper_decomposition', 'paper_understanding'],
    'Verification': ['technical_verification', 'data_source_validation', 'reproducibility_reasoning'],
    'Novelty': ['innovation_novelty', 'baseline_reproduction', 'validation_robustness'],
    'Report': ['scientific_writing', 'latex_generation', 'multi_stage_report', 'ieee_paper'],
    'Critique': ['adversarial_critique', 'hallucination_detection'],
    'Memory': ['memory_graph', 'citation_analysis'],
    'Visualization': ['visualization'],
    'Chatbot': ['interactive_chatbot', 'reviewer_style_critique', 'conversational'],
    'Scraper': ['data_scraper', 'web_scraper'],
    'News': ['news'],
    'Scoring': ['scoring'],
    'Planner': ['query_planner'],
    'Processing': ['data_cleaner'],
    'Re-Research': ['section_reresearch'],
    'Editor': ['editor']
}

for category, agents in categories.items():
    registered = [a for a in agents if a in AGENTS]
    if registered:
        print(f"\n{category}:")
        for a in registered:
            print(f"  - {a}")


Total Registered Agents: 38

Agents by Category:

Topic Discovery:
  - topic_discovery
  - topic_lock

Orchestration:
  - orchestrator

Discovery:
  - domain_intelligence
  - historical_review

Review:
  - slr
  - survey_meta_analysis

Synthesis:
  - gap_synthesis
  - research_question
  - conceptual_framework

Understanding:
  - paper_decomposition
  - paper_understanding

Verification:
  - technical_verification
  - data_source_validation
  - reproducibility_reasoning

Novelty:
  - innovation_novelty
  - baseline_reproduction
  - validation_robustness

Report:
  - scientific_writing
  - latex_generation
  - multi_stage_report
  - ieee_paper

Critique:
  - adversarial_critique
  - hallucination_detection

Memory:
  - memory_graph
  - citation_analysis

Visualization:
  - visualization

Chatbot:
  - interactive_chatbot
  - reviewer_style_critique
  - conversational

Scraper:
  - data_scraper
  - web_scraper

News:
  - news

Scoring:
  - scoring

Planner:
  - query_planner

Processing:


---

## 6. Agent Implementations

### 6.1 Topic Discovery Agents (Phase 0)


In [25]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.topic.agents import TopicDiscoveryAgent, TopicLockAgent, fallback_topic_suggestions

topic_discovery = TopicDiscoveryAgent()
topic_lock = TopicLockAgent()

print("TopicDiscoveryAgent:")
print(f"  Name: {topic_discovery.name}")
print(f"  Model: {topic_discovery.model_name}")

print("\nTopicLockAgent:")
print(f"  Name: {topic_lock.name}")

print("\nFallback Topic Suggestions:")
for i, t in enumerate(fallback_topic_suggestions("machine learning"), 1):
    print(f"  {i}. {t['title']}")


TopicDiscoveryAgent:
  Name: TopicDiscovery
  Model: openrouter/anthropic/claude-3.5-sonnet:beta

TopicLockAgent:
  Name: TopicLock

Fallback Topic Suggestions:
  1. A Comprehensive Survey on machine learning
  2. Applications of machine learning in Real-World Systems
  3. Recent Advances in machine learning: Methods, Benchmarks, and Open Challenges


### 6.2 Orchestrator Agent

In [26]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.orchestrator.orchestrator import OrchestratorAgent

orchestrator = OrchestratorAgent()
print("OrchestratorAgent:")
print(f"  Name: {orchestrator.name}")
print(f"  Routes: paper_analysis | literature_review")


OrchestratorAgent:
  Name: Orchestrator
  Routes: paper_analysis | literature_review


### 6.3 Discovery Agents

In [27]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.discovery.agents import DomainIntelligenceAgent, HistoricalReviewAgent

domain_intel = DomainIntelligenceAgent()
hist_review = HistoricalReviewAgent()

print("DomainIntelligenceAgent: {domain_intel.name}")
print("HistoricalReviewAgent: {hist_review.name}")


DomainIntelligenceAgent: {domain_intel.name}
HistoricalReviewAgent: {hist_review.name}


D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:69: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS() if DDGS else None
D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:69: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS() if DDGS else None
D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:487: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS() if DDGS else None
D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:449: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS() if DDGS else None
D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:69: RuntimeWarning: This package

### 6.4 Review Agents

In [28]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.review.agents import SystematicLiteratureReviewAgent, SurveyMetaAnalysisAgent

slr = SystematicLiteratureReviewAgent()
meta = SurveyMetaAnalysisAgent()

print("SystematicLiteratureReviewAgent: {slr.name}")
print("SurveyMetaAnalysisAgent: {meta.name}")


SystematicLiteratureReviewAgent: {slr.name}
SurveyMetaAnalysisAgent: {meta.name}


D:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\utils\providers.py:69: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  self.ddgs = DDGS() if DDGS else None


### 6.5 Understanding Agents (Pipeline B)

In [29]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.understanding.agents import PaperDecompositionAgent, PaperUnderstandingAgent

decomp = PaperDecompositionAgent()
understanding = PaperUnderstandingAgent()

print("PaperDecompositionAgent: {decomp.name}")
print("PaperUnderstandingAgent: {understanding.name}")


PaperDecompositionAgent: {decomp.name}
PaperUnderstandingAgent: {understanding.name}


### 6.6 Verification Agents

In [30]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.verification.agents import TechnicalVerificationAgent, DataSourceValidationAgent, ReproducibilityReasoningAgent

tech_verif = TechnicalVerificationAgent()
data_val = DataSourceValidationAgent()
repro = ReproducibilityReasoningAgent()

print("TechnicalVerificationAgent: {tech_verif.name}")
print("DataSourceValidationAgent: {data_val.name}")
print("ReproducibilityReasoningAgent: {repro.name}")


TechnicalVerificationAgent: {tech_verif.name}
DataSourceValidationAgent: {data_val.name}
ReproducibilityReasoningAgent: {repro.name}


### 6.7 Synthesis Agents

In [31]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.synthesis.agents import GapSynthesisAgent, ResearchQuestionEngineeringAgent, ConceptualFrameworkAgent

gap = GapSynthesisAgent()
rq = ResearchQuestionEngineeringAgent()
framework = ConceptualFrameworkAgent()

print("GapSynthesisAgent: {gap.name}")
print("ResearchQuestionEngineeringAgent: {rq.name}")
print("ConceptualFrameworkAgent: {framework.name}")


GapSynthesisAgent: {gap.name}
ResearchQuestionEngineeringAgent: {rq.name}
ConceptualFrameworkAgent: {framework.name}


### 6.8 Report Agents

In [32]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.report.agents import ScientificWritingAgent, LaTeXGenerationAgent

sci_writer = ScientificWritingAgent()
latex_gen = LaTeXGenerationAgent()

print("ScientificWritingAgent: {sci_writer.name}")
print("LaTeXGenerationAgent: {latex_gen.name}")


ScientificWritingAgent: {sci_writer.name}
LaTeXGenerationAgent: {latex_gen.name}


d:\SEM 6\AIML317_PROJECT_III\project_sgp\ai_engine\agents\report\latex_sanitizer.py:1: DeprecationWarning: invalid escape sequence '\d'
  """


### 6.9 Report Pipeline

In [34]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.report.pipeline import ReportPipeline, MultiStageReportAgent

pipeline = ReportPipeline()
msr = MultiStageReportAgent()

print("ReportPipeline: output_dir={pipeline.output_dir}")
print("MultiStageReportAgent: {msr.name}")


ReportPipeline: output_dir={pipeline.output_dir}
MultiStageReportAgent: {msr.name}


### 6.10 IEEE Paper Agent

In [35]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.report.ieee_paper import IEEEPaperAgent

ieee = IEEEPaperAgent()
print("IEEEPaperAgent: {ieee.name}")


IEEEPaperAgent: {ieee.name}


### 6.11 Novelty Agents

In [36]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.novelty.agents import InnovationNoveltyAgent, BaselineReproductionAgent, ValidationRobustnessAgent

innovation = InnovationNoveltyAgent()
baseline = BaselineReproductionAgent()
robustness = ValidationRobustnessAgent()

print("InnovationNoveltyAgent: {innovation.name}")
print("BaselineReproductionAgent: {baseline.name}")
print("ValidationRobustnessAgent: {robustness.name}")


InnovationNoveltyAgent: {innovation.name}
BaselineReproductionAgent: {baseline.name}
ValidationRobustnessAgent: {robustness.name}


### 6.12 Critique Agents

In [37]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.critique.agents import ReviewerAdversarialCritiqueAgent, HallucinationDetectionAgent

adversarial = ReviewerAdversarialCritiqueAgent()
hallucination = HallucinationDetectionAgent()

print("ReviewerAdversarialCritiqueAgent: {adversarial.name}")
print("HallucinationDetectionAgent: {hallucination.name}")


ReviewerAdversarialCritiqueAgent: {adversarial.name}
HallucinationDetectionAgent: {hallucination.name}


### 6.13 Visualization Agent

In [38]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.visualization.agents import VisualizationAgent

viz = VisualizationAgent()
print("VisualizationAgent: {viz.name}")


VisualizationAgent: {viz.name}


### 6.14 Memory Agents

In [39]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.memory.agents import MemoryKnowledgeGraphAgent, CitationGraphAnalysisAgent

memory = MemoryKnowledgeGraphAgent()
citation = CitationGraphAnalysisAgent()

print("MemoryKnowledgeGraphAgent: {memory.name}")
print("CitationGraphAnalysisAgent: {citation.name}")


MemoryKnowledgeGraphAgent: {memory.name}
CitationGraphAnalysisAgent: {citation.name}


### 6.15 Chatbot Agents

In [40]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.chatbot.agents import InteractivePaperChatbotAgent, ReviewerStyleCritiqueAgent

chatbot = InteractivePaperChatbotAgent()
reviewer = ReviewerStyleCritiqueAgent()

print("InteractivePaperChatbotAgent: {chatbot.name}")
print("ReviewerStyleCritiqueAgent: {reviewer.name}")


InteractivePaperChatbotAgent: {chatbot.name}
ReviewerStyleCritiqueAgent: {reviewer.name}


### 6.16 Query Planner Agent

In [41]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.planner.query_planner import QueryPlannerAgent

planner = QueryPlannerAgent()
print("QueryPlannerAgent: {planner.name}")
print("Modes: direct | search | deep")


QueryPlannerAgent: {planner.name}
Modes: direct | search | deep


### 6.17 Web Scraper Agent

In [42]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.scraper.web_scraper import WebScraperAgent

scraper = WebScraperAgent()
print("WebScraperAgent: {scraper.name}")


WebScraperAgent: {scraper.name}


### 6.18 Data Cleaner Agent

In [43]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.processing.cleaner import DataCleanerAgent

cleaner = DataCleanerAgent()
print("DataCleanerAgent: {cleaner.name}")


DataCleanerAgent: {cleaner.name}


### 6.19 Scoring Agent

In [44]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.scoring.agents import ScoringAgent

scoring = ScoringAgent()
print("ScoringAgent: {scoring.name}")


ScoringAgent: {scoring.name}


### 6.20 News Agent

In [45]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai_engine.agents.news.agent import NewsAgent

news = NewsAgent()
print("NewsAgent: {news.name}")


NewsAgent: {news.name}


---

## 7. Testing Code


In [46]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Unit Tests for BaseAgent
from ai_engine.agents.base import BaseAgent

agent = BaseAgent(name="Test", system_prompt="Test")

# Token Estimation
print("Token Estimation Tests:")
for text, expected_range in [
    ("Hello world", (1, 10)),
    ("This is a longer sentence", (5, 15)),
]:
    tokens = agent._estimate_tokens(text)
    status = "PASS" if expected_range[0] <= tokens <= expected_range[1] else "FAIL"
    print(f"  [{status}] '{text}' -> {tokens} tokens")


Token Estimation Tests:
  [PASS] 'Hello world' -> 2 tokens
  [PASS] 'This is a longer sentence' -> 6 tokens


In [47]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# JSON Extraction Tests
print("JSON Extraction Tests:")
for text in ['{"key": "value"}', '```json\n{"a":1}\n```', 'plain text']:
    result = agent._extract_json(text)
    has_json = isinstance(result, dict) and "raw_text" not in result
    print(f"  [{'PASS' if has_json else 'CHECK'}] {text[:30]}...")


[Test] Could not parse JSON. Returning raw text.


JSON Extraction Tests:
  [PASS] {"key": "value"}...
  [PASS] ```json
{"a":1}
```...
  [CHECK] plain text...


In [48]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Registry Validation
from ai_engine.agents.registry import AGENTS

print(f"Registry: {len(AGENTS)} agents registered")
lazy_count = sum(1 for a in AGENTS.values() if type(a).__name__ == '_LazyAgent')
print(f"LazyAgents: {lazy_count}")


Registry: 38 agents registered
LazyAgents: 38


---

## 8. Evaluation Code


In [49]:
# Setup Python path
import sys
from pathlib import Path
NOTEBOOK_DIR = Path('__file__').parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
AI_ENGINE_PATH = PROJECT_ROOT / 'ai_engine'
if str(AI_ENGINE_PATH) not in sys.path:
    sys.path.insert(0, str(AI_ENGINE_PATH))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Agent Evaluator Class
class AgentEvaluator:
    def __init__(self):
        self.results = {}
    
    def evaluate(self, agent_name, state):
        import time
        agent = AGENTS.get(agent_name)
        if not agent:
            return {"error": f"Agent {agent_name} not found"}
        
        start = time.time()
        try:
            result = agent.run(state)
            return {
                "agent": agent_name,
                "success": "error" not in result,
                "execution_time": time.time() - start,
                "keys": list(result.keys())
            }
        except Exception as e:
            return {"agent": agent_name, "error": str(e)}

evaluator = AgentEvaluator()
print("AgentEvaluator ready")


AgentEvaluator ready


In [55]:
# Quick Agent Benchmark
test_state = {"task": "machine learning", "_job_id": "test", "findings": {}}

for agent_name in ["topic_lock", "orchestrator"]:
    result = evaluator.evaluate(agent_name, test_state)
    print(f"\n{agent_name}:")
    if "error" in result:
        print(f"  Error: {result['error']}")
    else:
        print(f"  Success: {result.get('success')}")
        print(f"  Time: {result.get('execution_time', 0):.3f}s")


[Orchestrator] LLM suggested paper_analysis, but no paper URL was found. Forcing literature_review.


[TopicLock] Waiting for user topic selection...

topic_lock:
  Success: True
  Time: 0.000s

orchestrator:
  Success: True
  Time: 0.001s


---

## 9. System Architecture


In [56]:
# Pipeline Architecture
pipeline = {
    "Phase 0": ["topic_discovery", "topic_lock"],
    "Orchestration": ["orchestrator"],
    "Pipeline A": ["domain_intelligence", "slr", "gap_synthesis"],
    "Pipeline B": ["paper_decomposition", "paper_understanding"],
    "Report": ["scientific_writing", "latex_generation"]
}

print("LangGraph Pipeline:")
for stage, agents in pipeline.items():
    print(f"  {stage}: {', '.join(agents)}")


LangGraph Pipeline:
  Phase 0: topic_discovery, topic_lock
  Orchestration: orchestrator
  Pipeline A: domain_intelligence, slr, gap_synthesis
  Pipeline B: paper_decomposition, paper_understanding
  Report: scientific_writing, latex_generation


In [57]:
# Model Slots
print("Model Slots:")
for slot in ['MODEL_REASONING', 'MODEL_WRITING', 'MODEL_CODING', 'MODEL_CRITICAL']:
    print(f"  {slot}")


Model Slots:
  MODEL_REASONING
  MODEL_WRITING
  MODEL_CODING
  MODEL_CRITICAL


---

## 10. Summary

### Agent Count by Category

| Category | Count |
|----------|-------|
| Topic Discovery | 2 |
| Orchestration | 1 |
| Discovery | 2 |
| Review | 2 |
| Synthesis | 3 |
| Understanding | 2 |
| Verification | 3 |
| Novelty | 3 |
| Report | 4 |
| Critique | 2 |
| Memory | 2 |
| Visualization | 1 |
| Chatbot | 3 |
| Scraper | 2 |
| Supporting | 6 |

**Total: 35+ specialized AI agents**

### Key Features
1. **Lazy Loading**: Agents instantiated on first use
2. **Dual-Mode LLM**: OFFLINE/ONLINE/HYBRID
3. **Multi-Key Rotation**: Automatic API key rotation
4. **Deterministic Caching**: SHA-256 hash caching
5. **Token Management**: Automatic context truncation

### Usage Example
```python
from ai_engine.agents.registry import AGENTS

orchestrator = AGENTS['orchestrator']
result = orchestrator.run({"task": "Your topic", "_job_id": "123"})
```
